# 10 Flatten 与全连接分类头

上一节我们把 CNN 的整体结构串了一遍。

现在已经知道，CNN 前半部分通常负责提取图像特征，后半部分负责根据特征完成分类。

这一节专门学习后半部分：Flatten 和全连接分类头。

这一节仍然不写代码，只理解形状变化和每一步的作用。

## 1. 为什么需要 Flatten

卷积层和池化层处理的是图像结构。

所以它们输出的通常还是多维特征图。

比如一张 MNIST 图片经过几层卷积和池化后，可能得到：

$$
32\times7\times7
$$

这表示有 32 张特征图，每张大小是 7 x 7。

但是全连接层更像前面学过的 MLP，它接收的是一维向量。

所以在进入全连接层之前，需要把这些特征图拉平成一维向量。

这一步就叫 Flatten，中文可以叫展平。

## 2. Flatten 做了什么

Flatten 本身不学习参数。

它只是改变数据形状。

比如：

$$
32\times7\times7=1568
$$

所以：

```text
32 x 7 x 7 -> 1568
```

这表示原来有 32 张 7 x 7 的特征图。

展平后，变成一个长度为 1568 的特征向量。

注意，Flatten 不会改变数值本身，只是把它们重新排成一维。

## 3. Flatten 和 MLP 的关系

前面学 MLP 做 MNIST 时，我们把原始图片展平成：

$$
1\times28\times28 \rightarrow 784
$$

CNN 中也会展平。

但区别在于：

MLP 是一开始就展平原始图片。

CNN 是先卷积、池化，提取出特征图之后，再展平。

可以这样对比：

```text
MLP：原始图片 -> 展平 -> 分类

CNN：原始图片 -> 卷积/池化提特征 -> 展平 -> 分类
```

所以 CNN 的 Flatten 不是把原始图片直接打散，而是把已经提取好的特征图整理成分类头能接收的向量。

## 4. 什么是分类头

分类头可以理解成网络最后负责做分类判断的部分。

在入门 CNN 中，分类头通常由一个或多个全连接层组成。

它接收前面展平后的特征向量，然后输出每个类别的分数。

可以这样理解：

```text
卷积部分：负责看图，提取特征。
分类头：负责根据特征做判断。
```

分类头不是重新看图片，而是根据前面已经提取出来的特征进行分类。

## 5. 全连接层做什么

全连接层会把输入特征向量映射成新的向量。

比如展平后得到 1568 维特征。

可以先接一个全连接层，把它变成 128 维：

$$
1568 \rightarrow 128
$$

再接一个输出层，把 128 维变成类别数。

如果是 MNIST，类别数是 10：

$$
128 \rightarrow 10
$$

最终这 10 个数字就是 10 个类别的原始分数。

## 6. 什么是 logits

分类头最后输出的通常不是概率，而是 logits。

logits 可以先理解成模型对每个类别给出的原始分数。

比如 MNIST 有 10 个类别，输出可能是：

```text
[2.1, -0.3, 0.8, 4.5, 1.2, ...]
```

哪个类别的分数最高，模型就更倾向于认为图片属于哪个类别。

后面计算损失时，常用的交叉熵损失会处理这些 logits。

所以入门阶段先记住：

```text
分类头最后输出的是每个类别的原始分数。
```

## 7. 输出层神经元个数由什么决定

输出层神经元个数由任务的类别数决定。

如果是 MNIST，类别是 0 到 9，一共 10 类。

所以输出层是 10 个神经元。

如果是猫狗二分类，可以输出 2 个类别分数，也可以根据具体损失函数设计成 1 个输出。

如果是 CIFAR-10，也是 10 类，所以输出层通常也是 10 个神经元。

可以这样记：

```text
最终输出维度要和任务目标对得上。
```

## 8. 用 MNIST 看完整形状变化

假设前面卷积和池化后得到：

$$
32\times7\times7
$$

先展平：

$$
32\times7\times7=1568
$$

再进入第一个全连接层：

$$
1568 \rightarrow 128
$$

最后进入输出层：

$$
128 \rightarrow 10
$$

所以整个分类头可以理解成：

```text
32 x 7 x 7 -> 1568 -> 128 -> 10
```

最后的 10 对应 MNIST 的 10 个数字类别。

## 9. 加上 batch 后怎么变化

训练时通常是一批图片一起输入。

如果 batch size 是 64，卷积和池化后得到：

$$
64\times32\times7\times7
$$

Flatten 后变成：

$$
64\times1568
$$

第一个全连接层后：

$$
64\times128
$$

输出层后：

$$
64\times10
$$

注意，batch 维度 64 一直保留。

变化的是每个样本对应的特征维度。

## 10. Flatten 有没有参数

Flatten 没有可学习参数。

它只是改变形状。

比如：

```text
64 x 32 x 7 x 7 -> 64 x 1568
```

这个过程不会产生新的权重，也不会通过训练学习什么。

它只是把每个样本的多张特征图拉成一条长向量。

所以 Flatten 更像一个形状转换步骤，而不是一个学习层。

## 11. 全连接层有没有参数

全连接层有可学习参数。

比如一个全连接层：

$$
1568 \rightarrow 128
$$

它需要学习一组权重，把 1568 维特征映射成 128 维特征。

再比如输出层：

$$
128 \rightarrow 10
$$

它也有权重，用来把 128 维特征映射成 10 个类别分数。

所以：

```text
Flatten：无参数，只改形状。
全连接层：有参数，负责分类映射。
```

## 12. 为什么不能不展平直接全连接

全连接层通常把输入看成一维特征向量。

而卷积和池化输出的是多维特征图。

所以要先把每个样本的特征图整理成一条向量，再送进全连接层。

这个过程可以理解成：

```text
前面提取到的所有特征，整理成一份清单。
全连接层根据这份清单做分类判断。
```

Flatten 就是把这份清单整理出来。

## 13. 容易混淆的地方

第一，Flatten 不是池化。

池化会压缩高和宽，通常会减少特征图尺寸。

Flatten 只是把剩下的特征图拉成一维。

第二，Flatten 不会改变 batch size。

如果输入 batch 是 64，Flatten 后仍然是 64 个样本。

第三，分类头看到的不是原始图片。

它看到的是卷积和池化提取后的特征。

第四，输出层不是固定 10 个神经元。

输出层大小由具体任务类别数决定。

## 14. 本节小结

这一节先记住这些规则：

1. Flatten 是展平操作，用来把特征图拉成一维向量。
2. Flatten 没有可学习参数，只改变形状。
3. CNN 中的 Flatten 通常发生在卷积池化之后、全连接层之前。
4. 全连接分类头根据展平后的特征向量输出类别分数。
5. 分类头最后输出的通常是 logits，也就是每个类别的原始分数。
6. 输出层神经元个数由任务类别数决定。
7. 加上 batch 后，Flatten 只改变每个样本的特征维度，不改变 batch size。
8. CNN 是先提特征，再把特征整理成向量进行分类。

## 自检问题

1. Flatten 的作用是什么？
2. Flatten 有没有可学习参数？
3. 为什么 CNN 进入全连接层前通常需要 Flatten？
4. 32 x 7 x 7 展平后是多少维？
5. batch size 是 64，输入是 64 x 32 x 7 x 7，Flatten 后形状是什么？
6. 全连接分类头主要负责什么？
7. MNIST 最终输出为什么是 10 个分数？
8. logits 可以先理解成什么？
9. Flatten 和池化有什么区别？
10. CNN 中分类头看到的是原始图片还是提取后的特征？